# 2. Serving endpoint path

This notebook deploys a Docling model serving endpoint and shows the agent tool call pattern.

What it does:
- Registers a single-file Docling parsing model in Unity Catalog
- Creates or updates a serving endpoint
- Calls the endpoint by passing input paths and receiving output paths

In [ ]:
%pip install uv
%sh uv pip install .
%sh uv pip install ".[serving]"
%restart_python

In [ ]:
import mlflow
import tomllib

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

config = mlflow.models.ModelConfig(development_config="./config.yaml")
config = config.to_dict()

with open("pyproject.toml", "rb") as handle:
    pyproject = tomllib.load(handle)

optional_deps = pyproject.get("project", {}).get("optional-dependencies", {})
serving_deps = optional_deps.get("serving") or optional_deps.get("deployment") or []

MODEL_NAME = f"{config['catalog']}.{config['schema']}.docling_parser"
ENDPOINT_NAME = config["serving_endpoint"]


def register_model() -> str:
    mlflow.set_registry_uri("databricks-uc")

    with mlflow.start_run(run_name="docling_endpoint_registration"):
        from docling_endpoint import DoclingParsingModel

        model_info = mlflow.pyfunc.log_model(
            artifact_path="docling_parser",
            python_model=DoclingParsingModel(),
            code_paths=["docling_endpoint.py"],
            conda_env={
                "channels": ["conda-forge"],
                "dependencies": [
                    "python=3.12.3",
                    "pip",
                    {
                        "pip": serving_deps
                    },
                ],
            },
            pip_requirements=serving_deps,
        )

    return model_info.model_uri


def ensure_endpoint(model_name: str, model_version: str, endpoint_name: str) -> None:
    client = WorkspaceClient()

    served_entity = ServedEntityInput(
        name="docling-parser",
        entity_name=model_name,
        entity_version=str(model_version),
        workload_size="Small",
        workload_type="GPU_SMALL",
        scale_to_zero_enabled=True,
    )

    try:
        client.serving_endpoints.get(endpoint_name)
        client.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[served_entity],
        )
    except Exception:
        client.serving_endpoints.create_and_wait(
            name=endpoint_name,
            config=EndpointCoreConfigInput(served_entities=[served_entity]),
        )


model_uri = register_model()
registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
ensure_endpoint(MODEL_NAME, registered_model.version, ENDPOINT_NAME)
print(f"Serving endpoint ready: {ENDPOINT_NAME}")

In [ ]:
from mlflow.deployments import get_deploy_client

sample_inputs = config.get(
    "evaluate_sample_inputs",
    [f"/Volumes/{config['catalog']}/{config['schema']}/{config['input_volume']}/sample.pdf"],
)
output_root = f"/Volumes/{config['catalog']}/{config['schema']}/{config['output_volume']}"
options = {"generate_page_images": False}

rows = [[path, output_root, options] for path in sample_inputs]
request = {
    "dataframe_split": {
        "columns": ["file_path", "output_root", "options"],
        "data": rows,
    }
}

deploy_client = get_deploy_client("databricks")
response = deploy_client.predict(endpoint=ENDPOINT_NAME, inputs=request)
predictions = response.get("predictions", response)

output_paths = [
    item["output_path"]
    for item in predictions
    if item.get("status") == "success" and item.get("output_path")
]

print("Parsed output paths:")
for path in output_paths:
    print(path)